# AIOps: Time-Series Forecasting (Prophet)

In [ ]:
# Day 43 — AIOps: Time-Series Forecasting
#
# AIOps = Artificial Intelligence for IT Operations.
# It means using ML to solve DevOps problems.
#
# The Scenario:
# You monitor CPU usage across your Kubernetes cluster. You notice spikes every Friday.
# Instead of waiting for the CPU to hit 90% (and lagging the app) before auto-scaling,
# what if you could PREDICT that tomorrow at 5 PM you will need 10 more nodes?
#
# To do this, we use Time-Series Forecasting.
# The most popular tool for this is 'Prophet' (created by Meta/Facebook).
# It is designed to handle daily/weekly/yearly seasonality automatically.
#
# Today we'll cover:
#   1. Formatting data for Prophet (ds and y)
#   2. Training a Prophet model on server metrics
#   3. Predicting future server load
#
# Install: pip install prophet pandas matplotlib

### 1. Generating Mock Server Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Let's generate 60 days of historical CPU usage data
dates = pd.date_range(start="2024-01-01", periods=60, freq="D")

# Create a pattern: Base load + Weekend drop + Random noise
cpu_load = []
for d in dates:
    base = 50
    # Weekends (Saturday=5, Sunday=6) have lower traffic
    if d.weekday() >= 5:
        base -= 20
    # Add some random noise
    noise = np.random.normal(0, 5)
    # Add a slight upward trend over time (growing user base)
    trend = (d - dates[0]).days * 0.2
    
    cpu_load.append(base + noise + trend)

# PROPHET RULE: The dataframe MUST have columns named 'ds' (datestamp) and 'y' (value)
df = pd.DataFrame({
    'ds': dates,
    'y': cpu_load
})

print(df.head())

# Let's plot our historical data
plt.figure(figsize=(10, 4))
plt.plot(df['ds'], df['y'])
plt.title("Historical Server CPU Load (60 Days)")
plt.ylabel("CPU %")
plt.grid(True)
plt.savefig('historical_cpu.png')
print("\nPlot saved as historical_cpu.png")

### 2. Training the Prophet Model

In [ ]:
# To run this cell, you need `prophet` installed.
# Prophet is incredibly easy to use. It follows the sklearn fit/predict pattern.

training_code = '''from prophet import Prophet

# 1. Initialize the model
# We tell it to look for weekly seasonality (since weekends drop)
m = Prophet(yearly_seasonality=False, daily_seasonality=False, weekly_seasonality=True)

# 2. Fit the model to our historical CPU data
m.fit(df)
print("Model trained!")
'''

print("Prophet Training Pattern:")
print(training_code)

### 3. Predicting Future Server Load

In [ ]:
# Now we ask Prophet to predict the NEXT 14 days of CPU usage.

prediction_code = '''
# 1. Create a dataframe with future dates
future = m.make_future_dataframe(periods=14)  # Add 14 days to the end

# 2. Predict!
forecast = m.predict(future)

# The forecast dataframe contains a lot of info. The most important are:
# 'ds' (date), 'yhat' (predicted value), 'yhat_lower' (lower bound), 'yhat_upper' (upper bound)
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

# 3. Plot the prediction (Prophet has built-in plotting)
fig1 = m.plot(forecast)
fig1.savefig('cpu_forecast.png')

# 4. Plot the components (Trend vs Weekly Seasonality)
fig2 = m.plot_components(forecast)
fig2.savefig('cpu_components.png')
'''

print("Prophet Prediction Pattern:")
print(prediction_code)

print("\nWhat AIOps enables:")
print("If 'yhat_upper' > 85% for next Tuesday, your Python script can trigger")
print("Azure SDK to provision 5 more VMs on Monday night, ensuring zero downtime.")

In [ ]:
# Practice Questions:
#
# 1. Why must the columns be named 'ds' and 'y' for Prophet?
# 2. What is the difference between 'yhat' and 'yhat_upper'? Why would an SRE
#    care more about 'yhat_upper' when provisioning capacity?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The Predictive Auto-Scaler
# Write pseudo-code for an Airflow DAG that:
# 1. Downloads the last 30 days of CPU metrics from Datadog/Azure Monitor.
# 2. Formats it into a 'ds' and 'y' DataFrame.
# 3. Uses Prophet to predict 'yhat_upper' for tomorrow.
# 4. If 'yhat_upper' > 80, calls an API to scale up the Kubernetes cluster.

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Proactive vs Reactive: Standard DevOps reacts to CPU spikes (causing 5 mins of lag while scaling).
#    AIOps scales BEFORE the spike happens.
# 2. Cost Savings: You can also predict when traffic will drop (holidays/weekends) and scale down safely.